# FineWeb-Edu 20M baseline — persistent Colab run

Select a GPU runtime, then run all cells. The notebook clones the public `main` branch from GitHub into ephemeral Colab storage and uses Google Drive only as the durable artifact store. It performs a fresh-process baseline batch-capacity search with 10% reserved-VRAM headroom, freezes the largest repeatedly successful physical batch, and only then runs smoke and full training. Data, capacity evidence, live logs, structured metrics, checkpoints, frozen configs, and summaries are written under `MyDrive/Reversability/artifacts`. Re-running the notebook refreshes the source clone and resumes an incomplete run automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = 'https://github.com/LokeshJatangi/Reversability.git'
REPO_BRANCH = 'main'
REPO = Path('/content/Reversability')
PERSIST_ROOT = Path('/content/drive/MyDrive/Reversability')
ARTIFACTS = PERSIST_ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SESSION_LOG = ARTIFACTS / 'colab-session.log'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)
if not (REPO / 'requirements.txt').is_file():
    raise FileNotFoundError(f'Cloned repository is incomplete: {REPO}')

def run_logged(command, log_path=SESSION_LOG, cwd=REPO):
    command = [str(item) for item in command]
    stamp = datetime.now(timezone.utc).isoformat()
    with Path(log_path).open('a', encoding='utf-8') as log:
        header = f'\n[{stamp}] $ ' + ' '.join(command) + '\n'
        print(header, end='')
        log.write(header); log.flush()
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush()
        code = process.wait()
        os.fsync(log.fileno())
    if code:
        raise subprocess.CalledProcessError(code, command)

print('Repository:', REPO)
print('Persistent artifacts:', ARTIFACTS)


In [ ]:
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No GPU runtime detected. Select Runtime > Change runtime type > GPU.')
run_logged(['nvidia-smi'])
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO / 'requirements.txt'])
run_logged([sys.executable, '-m', 'pytest', '-q'])
run_logged([sys.executable, '-m', 'pip', 'freeze'], ARTIFACTS / 'pip-freeze.log')


In [ ]:
DATA_NAME = 'data_fineweb_edu_gpt2_50m_v1'
PERSIST_DATA = ARTIFACTS / 'data' / DATA_NAME
LOCAL_DATA = Path('/content/reversibility-data') / DATA_NAME
REPO_DATA = REPO / 'data' / DATA_NAME

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b''): digest.update(chunk)
    return digest.hexdigest()

def validate_dataset(root):
    root = Path(root)
    manifest = json.loads((root / 'manifest.json').read_text())
    assert manifest['training_targets'] == 50_000_000
    assert manifest['validation_targets'] == 1_000_000
    for name, targets in (('train', 50_000_000), ('validation', 1_000_000)):
        path = root / f'{name}.bin'
        assert path.stat().st_size == 2 * (targets + 1), name
        assert sha256_file(path) == manifest['files'][name]['sha256'], name
    for name, record in manifest['tokenizer']['files'].items():
        path = root / 'tokenizer' / name
        assert path.stat().st_size == record['bytes'], name
        assert sha256_file(path) == record['sha256'], name
    return manifest

persistent_valid = False
if (PERSIST_DATA / 'manifest.json').is_file():
    try:
        validate_dataset(PERSIST_DATA)
        persistent_valid = True
    except Exception as error:
        print('Persistent dataset failed validation:', repr(error))
if PERSIST_DATA.exists() and not persistent_valid:
    archive = ARTIFACTS / 'incomplete' / f"{DATA_NAME}-{int(time.time())}"
    archive.parent.mkdir(parents=True, exist_ok=True)
    PERSIST_DATA.rename(archive)
    print('Preserved incomplete or invalid dataset at', archive)

if not (PERSIST_DATA / 'manifest.json').is_file():
    if (REPO_DATA / 'manifest.json').is_file():
        validate_dataset(REPO_DATA)
        stage = REPO_DATA
        print('Using the verified dataset bundled with the repository.')
    else:
        stage = Path('/content/fineweb-build') / DATA_NAME
        if stage.exists(): shutil.rmtree(stage)
        try:
            run_logged([sys.executable, '-u', REPO / 'scripts/prepare_fineweb_edu.py',
                        '--output-dir', stage, '--train-targets', '50000000',
                        '--validation-targets', '1000000'], ARTIFACTS / 'dataset-preparation.log')
        except subprocess.CalledProcessError:
            if not (stage / 'manifest.json').is_file(): raise
            print('Preparation exited after writing a manifest; accepting only if full hash validation passes.')
        validate_dataset(stage)
    PERSIST_DATA.parent.mkdir(parents=True, exist_ok=True)
    drive_stage = PERSIST_DATA.with_name(PERSIST_DATA.name + f'.staging-{int(time.time())}')
    shutil.copytree(stage, drive_stage)
    validate_dataset(drive_stage)
    drive_stage.rename(PERSIST_DATA)

if LOCAL_DATA.exists(): shutil.rmtree(LOCAL_DATA)
LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(PERSIST_DATA, LOCAL_DATA)
manifest = validate_dataset(LOCAL_DATA)
print('Frozen dataset verified and copied to local runtime:', LOCAL_DATA)


In [ ]:
CONFIG_DIR = ARTIFACTS / 'configs'
RUNS_DIR = ARTIFACTS / 'runs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
source_files = [p for p in REPO.rglob('*') if p.is_file() and not any(part in {'.git', '__pycache__', '.pytest_cache', 'artifacts', 'data', 'runs', 'graphify-out'} for part in p.parts)]
source_manifest = {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(source_files)}
source_manifest_text = json.dumps(source_manifest, indent=2) + '\n'
source_manifest_sha = hashlib.sha256(source_manifest_text.encode()).hexdigest()
(ARTIFACTS / 'source-manifest.json').write_text(source_manifest_text)
snapshot_dir = ARTIFACTS / 'source-snapshots'
snapshot_dir.mkdir(parents=True, exist_ok=True)
snapshot_path = snapshot_dir / f'{source_manifest_sha}.zip'
if not snapshot_path.exists():
    with zipfile.ZipFile(snapshot_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in source_files: archive.write(path, path.relative_to(REPO))

base = json.loads((REPO / 'configs/baseline.json').read_text())
base_config_sha = hashlib.sha256((REPO / 'configs/baseline.json').read_bytes()).hexdigest()
assert base['max_train_targets'] == 50_000_000
assert base['precision'] == 'fp16'
nominal_effective_batch = base['physical_batch_size'] * base['gradient_accumulation_steps']
capacity_dir = ARTIFACTS / 'benchmarks'
capacity_dir.mkdir(parents=True, exist_ok=True)
CAPACITY_REPORT = capacity_dir / 'baseline_batch_capacity.json'
existing_full_config = CONFIG_DIR / 'baseline_colab.json'
existing_full_run = RUNS_DIR / 'baseline_20m_fineweb_edu_50m_v1' / 'latest.pt'
import torch
current_gpu = torch.cuda.get_device_name(0)
if existing_full_run.exists() and existing_full_config.exists():
    frozen = json.loads(existing_full_config.read_text())
    selected_physical_batch = frozen['physical_batch_size']
    selected_accumulation = frozen['gradient_accumulation_steps']
    if CAPACITY_REPORT.exists():
        frozen_gpu = json.loads(CAPACITY_REPORT.read_text()).get('gpu_name')
        if frozen_gpu and frozen_gpu != current_gpu:
            raise RuntimeError(f'Resume requires the frozen GPU type {frozen_gpu}; current GPU is {current_gpu}')
    print('Using batch configuration bound to existing checkpoint.')
else:
    capacity_record = json.loads(CAPACITY_REPORT.read_text()) if CAPACITY_REPORT.exists() else {}
    report_is_current = (capacity_record.get('gpu_name') == current_gpu and
                         capacity_record.get('config_sha256') == base_config_sha)
    if not report_is_current:
        run_logged([sys.executable, '-u', REPO / 'scripts/benchmark_batch_capacity.py',
                    '--config', REPO / 'configs/baseline.json', '--output', CAPACITY_REPORT,
                    '--max-batch', '256', '--repetitions', '3',
                    '--headroom-fraction', '0.10'], capacity_dir / 'baseline_batch_capacity.log')
    capacity = json.loads(CAPACITY_REPORT.read_text())
    assert capacity['gpu_name'] == current_gpu
    selected_physical_batch = capacity['selected_physical_batch_size']
    candidates = {max(1, nominal_effective_batch // selected_physical_batch),
                  max(1, -(-nominal_effective_batch // selected_physical_batch))}
    selected_accumulation = min(candidates, key=lambda value: abs(selected_physical_batch * value - nominal_effective_batch))
base['physical_batch_size'] = selected_physical_batch
base['gradient_accumulation_steps'] = selected_accumulation
base['capacity_search_report'] = str(CAPACITY_REPORT)
base['data_dir'] = str(LOCAL_DATA)
base['checkpoint_interval'] = 100
base['source_manifest_sha256'] = source_manifest_sha

def freeze_config(path, config):
    text = json.dumps(config, indent=2) + '\n'
    if path.exists() and path.read_text() != text:
        output = Path(json.loads(path.read_text())['output_dir'])
        if (output / 'latest.pt').exists():
            raise RuntimeError(f'Refusing to change a config with a resumable checkpoint: {path}')
    path.write_text(text)

full = dict(base)
full['output_dir'] = str(RUNS_DIR / 'baseline_20m_fineweb_edu_50m_v1')
FULL_CONFIG = CONFIG_DIR / 'baseline_colab.json'
freeze_config(FULL_CONFIG, full)

smoke = dict(base)
smoke['experiment_name'] = 'baseline_20m_fineweb_edu_50m_v1_smoke'
smoke['output_dir'] = str(RUNS_DIR / smoke['experiment_name'])
smoke['eval_interval'] = 2
smoke['checkpoint_interval'] = 2
SMOKE_CONFIG = CONFIG_DIR / 'baseline_smoke_colab.json'
freeze_config(SMOKE_CONFIG, smoke)

print('Frozen Colab configs and source snapshot saved:', snapshot_path)
print('Physical batch:', selected_physical_batch, 'Accumulation:', selected_accumulation,
      'Effective batch:', selected_physical_batch * selected_accumulation)


In [ ]:
def run_or_resume(config_path, target_budget, extra_args=()):
    config = json.loads(Path(config_path).read_text())
    output = Path(config['output_dir'])
    output.mkdir(parents=True, exist_ok=True)
    summary_path = output / 'run_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary.get('completed_full_target_budget') and summary.get('committed_targets') == target_budget:
            print('Already complete:', config['experiment_name'])
            return summary
    command = [sys.executable, '-u', REPO / 'scripts/train_baseline.py', '--config', config_path, '--device', 'cuda', *extra_args]
    latest = output / 'latest.pt'
    if latest.exists(): command.extend(['--resume', latest])
    run_logged(command, output / 'console.log')
    summary = json.loads(summary_path.read_text())
    if not summary.get('completed_full_target_budget') or summary.get('committed_targets') != target_budget:
        raise RuntimeError(f"Run did not complete its target budget: {summary}")
    return summary

smoke_summary = run_or_resume(SMOKE_CONFIG, 65_536, ('--max-targets', '65536'))
assert smoke_summary['committed_targets'] == 65_536
assert smoke_summary['final_validation_loss'] is not None
smoke_metrics = [json.loads(line) for line in (Path(smoke['output_dir']) / 'metrics.jsonl').read_text().splitlines()]
rates = [row['end_to_end_targets_per_second_this_process'] for row in smoke_metrics if row.get('event') == 'training_progress' and row.get('end_to_end_targets_per_second_this_process', 0) > 0]
if rates:
    projected_minutes = 50_000_000 / rates[-1] / 60
    print(f'Smoke training-rate projection (rough, excludes full-run validation/checkpoint effects): {projected_minutes:.1f} minutes')
print('Smoke accepted:', smoke_summary)


In [ ]:
full_summary = run_or_resume(FULL_CONFIG, 50_000_000)
assert full_summary['committed_targets'] == 50_000_000
print('FULL BASELINE COMPLETE:', json.dumps(full_summary, indent=2))


In [ ]:
inventory = []
for path in sorted(ARTIFACTS.rglob('*')):
    if path.is_file(): inventory.append({'path': str(path.relative_to(ARTIFACTS)), 'bytes': path.stat().st_size})
(ARTIFACTS / 'artifact-inventory.json').write_text(json.dumps(inventory, indent=2) + '\n')
print(json.dumps(inventory, indent=2))
print('All artifacts remain in:', ARTIFACTS)
print('Use Google Drive for desktop to mirror this fixed folder to the laptop automatically; no Colab download dialog is needed.')
